In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
#simple text search using RAG
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

from rag_helper import RAGBase

## RAG using LLM
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes — you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'

In [4]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [5]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
#embedding of all documents
import numpy as np
from tqdm.auto import tqdm

texts = []
for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)


batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)


X = np.array(vectors)
X.shape

  0%|          | 0/27 [00:00<?, ?it/s]

(1350, 384)

In [7]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [8]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [10]:
vector_assistant.rag("do we have Data engineering courses?")

"I don't know."